In [1]:
import pandas as pd
import warnings

# Вимикаю всі попередження (warnings) - щоб червоного було менше
warnings.filterwarnings('ignore')

In [2]:
df1 = pd.read_csv('C:\\Users\\admin\\Downloads\\stack-overflow-developer-survey-2025\\survey_results_public.csv')
df2 = pd.read_csv('C:\\Users\\admin\\Downloads\\stack-overflow-developer-survey-2025\\survey_results_schema.csv')

In [3]:
#налаштування відобращення всіх колонок в датафреймі
pd.set_option('display.max_columns', None)

In [4]:
print('survey_results_public: ')
print(df1.head())

survey_results_public: 
   ResponseId                      MainBranch              Age  \
0           1  I am a developer by profession  25-34 years old   
1           2  I am a developer by profession  25-34 years old   
2           3  I am a developer by profession  35-44 years old   
3           4  I am a developer by profession  35-44 years old   
4           5  I am a developer by profession  35-44 years old   

                                           EdLevel  \
0  Master’s degree (M.A., M.S., M.Eng., MBA, etc.)   
1              Associate degree (A.A., A.S., etc.)   
2     Bachelor’s degree (B.A., B.S., B.Eng., etc.)   
3     Bachelor’s degree (B.A., B.S., B.Eng., etc.)   
4  Master’s degree (M.A., M.S., M.Eng., MBA, etc.)   

                                          Employment  \
0                                           Employed   
1                                           Employed   
2  Independent contractor, freelancer, or self-em...   
3                             

In [5]:
print('survey_results_schema :')
print(df2.head())

survey_results_schema :
     qid          qname                                           question  \
0  QID18  TechEndorse_1  What attracts you to a technology or causes yo...   
1  QID18  TechEndorse_2  What attracts you to a technology or causes yo...   
2  QID18  TechEndorse_3  What attracts you to a technology or causes yo...   
3  QID18  TechEndorse_4  What attracts you to a technology or causes yo...   
4  QID18  TechEndorse_5  What attracts you to a technology or causes yo...   

  type                                      sub  sq_id  
0   RO  AI integration or AI Agent capabilities    1.0  
1   RO                          Easy-to-use API    2.0  
2   RO                  Robust and complete API    3.0  
3   RO     Customizable and manageable codebase    4.0  
4   RO                   Reputation for quality    5.0  


In [6]:
# Детальна перевірка табл.
print('--- Технічна інформація про таблицю 1---')
print(df1.info())

print('--- Технічна інформація про таблицю 2---')
print(df2.info())

--- Технічна інформація про таблицю 1---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Columns: 172 entries, ResponseId to JobSat
dtypes: float64(52), int64(1), object(119)
memory usage: 64.6+ MB
None
--- Технічна інформація про таблицю 2---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   qid       139 non-null    object 
 1   qname     139 non-null    object 
 2   question  139 non-null    object 
 3   type      139 non-null    object 
 4   sub       49 non-null     object 
 5   sq_id     49 non-null     float64
dtypes: float64(1), object(5)
memory usage: 6.6+ KB
None


## Завдання 1. Підрахунок загальної кількості респондентів

In [7]:
#Визначаю загальну кіль-ть респондентів, які взяли участь в опитуванні Stack Overflow 2025.
total_respondents = df1['ResponseId'].nunique()
print(f"Загальна кількість респондентів: {total_respondents}")

Загальна кількість респондентів: 49191


## Завдання 2. Аналіз повноти відповідей респондентів

In [8]:
# 1. Отримую множину питань зі схеми
schema_questions = set(df2['qname'])

# 2. Отримую множину назв колонок з основного датасету
public_columns = set(df1.columns)

# 3. Інтерсекція множин (знаходжу спільні назви)
# Це важливо, щоб не шукати відповіді в технічних колонках, яких немає в опитуванні
common_questions = list(schema_questions.intersection(public_columns))

print(f"Кількість питань для перевірки після інтерсекції: {len(common_questions)}")

# 4. Створюю df - підмножину даних лише з цими питаннями
#Підмножина - шматок табл., частина набору даних
df_questions = df1[common_questions]

# 5. Шукаю респондентів, у яких НЕМАЄ жодного пропущеного значення (NaN) у цих колонках
# dropna() видалить всі рядки, де є хоча б один NaN
# len() підрахує тих, хто залишився
perfect_respondents_count = len(df_questions.dropna())

print(f'Кількість респондентів, які відповіли на всі запитання: {perfect_respondents_count}')

Кількість питань для перевірки після інтерсекції: 126
Кількість респондентів, які відповіли на всі запитання: 0


* результат «0» — це нормально і не є помилкою:
1) забагато питань в опитуванні, респонденти можуть пропускати необов'язкові
2) якщо людина відповідає, що вона не використовує ШІ, Pd автоматично ставить NaN у всіх наступних 20 питаннях про ШІ.

## Завдання 3. Статистичний аналіз досвіду респондентів

In [9]:
print(df1['WorkExp'])

0         8.0
1         2.0
2        10.0
3         4.0
4        21.0
         ... 
49186     NaN
49187    25.0
49188    17.0
49189     2.0
49190     NaN
Name: WorkExp, Length: 49191, dtype: float64


In [10]:
# Обчислюю міри центральної тенденції
mean = round(df1['WorkExp'].mean(), 1)
median = round(df1['WorkExp'].median(), 1)
mode = round(df1['WorkExp'].mode()[0], 1)
#Я використала індексацію [0],
#щоб отримати конкретне числове значення моди з об'єкта Series, який повертає метод .mode()
print(f'Міри центральної тенденції для поля WorkExp: mean = {mean}, median = {median}, mode = {mode}')

Міри центральної тенденції для поля WorkExp: mean = 13.4, median = 10.0, mode = 10.0


* Mean (13.4) vs Median (10.0). 
Це класичний приклад додатно-асиметричного розподілу (або розподілу з правим хвостом).
* Середнє 13.4 вище за медіану, бо його "тягнуть" вгору досвідчені фахівці (ті, хто в професії 30+ років). Їх менше, але їхні великі числа сильно впливають на середнє арифметичне.
* Мода 10 збігається з медіаною — це свідчить про те, що 10 років досвіду є певним "плато" або найпоширенішою точкою в кар'єрі опитаних.

In [11]:
# Створюю об'єкт Series для фінального результату
work_exp_stats = pd.Series({
    'Mean': mean,
    'Median': median,
    'Mode': mode
}, name="WorkExp_Statistics")

print(work_exp_stats)

Mean      13.4
Median    10.0
Mode      10.0
Name: WorkExp_Statistics, dtype: float64


## Завдання 4. Аналіз віддаленої роботи

In [12]:
print(df1['RemoteWork'])

0                                                   Remote
1        Hybrid (some in-person, leans heavy to flexibi...
2                                                      NaN
3                                                   Remote
4                                                      NaN
                               ...                        
49186                                                  NaN
49187                                                  NaN
49188    Hybrid (some in-person, leans heavy to flexibi...
49189                                                  NaN
49190                                                  NaN
Name: RemoteWork, Length: 49191, dtype: object


In [13]:
Remote_respondents = len(df1.loc[df1['RemoteWork'].str.lower() == 'remote'])
#Перетворюю всі значення в колонці на маленькі літери під час перевірки
print(f'Кількість респондентів, які працюють віддалено: {Remote_respondents}')

Кількість респондентів, які працюють віддалено: 10931


## Завдання 5. Визначення популярності Python

In [14]:
#Створюю новий стовпець worked_with_python (boolean type)
def check_python(text):
    # Якщо це текст і в ньому є слово python (ігноруючи регістр)
    if isinstance(text, str) and 'python' in text.lower():
        return True
    return False

df1['worked_with_python'] = df1['LanguageHaveWorkedWith'].apply(check_python)

# Рахую % (кіль-ть True / загальна кількість відповідей) * 100
#True = 1 , також ділю на count() оригінальної колонки, 
#щоб врахувати тільки тих респондентів, які реально надали відповідь на це питання, 
#і уникнути спотворення статистики через пропущені значення (NaN)
python_percentage = (df1['worked_with_python'].sum() / df1['LanguageHaveWorkedWith'].count() * 100).round(1)

print(f"Відсоток респондентів, які програмують на Python: {python_percentage}%")

Відсоток респондентів, які програмують на Python: 58.3%


## Завдання 6. Аналіз шляхів навчання програмуванню

In [15]:
print(df1['LearnCode'])

0        Online Courses or Certification (includes all ...
1        Online Courses or Certification (includes all ...
2        Online Courses or Certification (includes all ...
3        Other online resources (e.g. standard search, ...
4                                                      NaN
                               ...                        
49186                                                  NaN
49187    Online Courses or Certification (includes all ...
49188    Online Courses or Certification (includes all ...
49189    Videos (not associated with specific online co...
49190    Online Courses or Certification (includes all ...
Name: LearnCode, Length: 49191, dtype: object


In [16]:
# 1. Створюю новий стовпець (булевого типу: True/False)
#case=False Ігнорує регістр
df1['online_learners'] = df1['LearnCode'].str.contains('Online Courses', case=False, na=False)

# 2. Рахую кіль-ть True 
# Використовую метод .sum() від Pd — це швидше і професійніше за звичайний sum()
online_count = df1['online_learners'].sum()

print(f"Кількість респондентів, які навчалися через онлайн-курси: {online_count}")

Кількість респондентів, які навчалися через онлайн-курси: 10973


## Завдання 7. Географічний аналіз компенсації Python-розробників

In [17]:
print(df1['ConvertedCompYearly'])

0         61256.0
1        104413.0
2         53061.0
3         36197.0
4         60000.0
           ...   
49186         NaN
49187         NaN
49188         NaN
49189         NaN
49190         NaN
Name: ConvertedCompYearly, Length: 49191, dtype: float64


In [18]:
# 1. Фільтрую: тільки Python-розробники та ті, у кого вказана зарплата
# Використ. стовпець worked_with_python з Завдання 5
python_salary_df = df1[(df1['worked_with_python'] == True ) & (df1['ConvertedCompYearly'].notna())]
# 2. Групую за країною та рахую статистику
python_salary_df = python_salary_df.groupby('Country')['ConvertedCompYearly'].agg(['mean', 'median']).round(1)
#3. Знаходжу країну з max середнім
top_country_name = python_salary_df['mean'].idxmax()
top_country_value = python_salary_df['mean'].max()
#4. Знаходжу країну з min середнім
anti_top_country_name = python_salary_df['mean'].idxmin()
anti_top_country_value = python_salary_df['mean'].min()
print(f'Країна з максимальним середнім значенням по компенс. {top_country_name} ({top_country_value:.1f})')
print(f'Країна з мінімальним середнім значенням по компенс. {anti_top_country_name} ({anti_top_country_value:.1f})')
print(python_salary_df)

Країна з максимальним середнім значенням по компенс. Oman (390135.0)
Країна з мінімальним середнім значенням по компенс. Antigua and Barbuda (1.0)
                                          mean    median
Country                                                 
Afghanistan                            22328.7    1000.0
Albania                                47217.6   50000.0
Algeria                                20187.3    7088.0
Andorra                               226103.5  226103.5
Antigua and Barbuda                        1.0       1.0
...                                        ...       ...
Venezuela, Bolivarian Republic of...    9908.6    3000.0
Viet Nam                              218837.2    8254.0
Yemen                                  32929.5   23672.0
Zambia                                  5424.2    3206.0
Zimbabwe                               34000.0   25500.0

[153 rows x 2 columns]


## Завдання 8. Аналіз освіти найбільш оплачуваних спеціалістів

In [19]:
# 1. Сортую за спаданням
top_5 = df1.sort_values(by = 'ConvertedCompYearly', ascending = False)[:5]
print(top_5[['ResponseId', 'EdLevel', 'ConvertedCompYearly']])

       ResponseId                                          EdLevel  \
34267       34268              Associate degree (A.A., A.S., etc.)   
28700       28701  Master’s degree (M.A., M.S., M.Eng., MBA, etc.)   
43143       43144              Associate degree (A.A., A.S., etc.)   
35353       35354     Bachelor’s degree (B.A., B.S., B.Eng., etc.)   
45971       45972  Master’s degree (M.A., M.S., M.Eng., MBA, etc.)   

       ConvertedCompYearly  
34267           50000000.0  
28700           33552715.0  
43143           18387548.0  
35353           15430267.0  
45971           13921760.0  


* Цікаво, що серед топ-5 найбільш оплачуваних спеціалістів лише двоє мають ступінь магістра, що підкреслює гнучкість IT-індустрії до формальної освіти.

## *Завдання 9. Аналіз популярності Python по віковим категоріям 

In [20]:
print(df1['Age'])

0        25-34 years old
1        25-34 years old
2        35-44 years old
3        35-44 years old
4        35-44 years old
              ...       
49186    18-24 years old
49187    45-54 years old
49188    35-44 years old
49189    25-34 years old
49190    18-24 years old
Name: Age, Length: 49191, dtype: object


In [21]:
# Групуюю по віку і рахую середнє (частку знавців Python)
# Оскільки True = 1, а False = 0, середнє якраз і буде часткою Python-істів
age_stats = df1.groupby('Age')['worked_with_python'].mean() * 100

# Перетворюю в DataFrame
age_stats_df = age_stats.reset_index()
age_stats_df['worked_with_python'] = age_stats_df['worked_with_python'].round(1)
print(age_stats_df)

                 Age  worked_with_python
0    18-24 years old                40.0
1    25-34 years old                36.9
2    35-44 years old                36.7
3    45-54 years old                38.6
4    55-64 years old                37.2
5  65 years or older                31.6
6  Prefer not to say                31.2


* Найвищий відсоток (40%) саме у молоді (18-24 роки).
  Це підтверджує, що Python зараз є основною мовою для навчання та входу в IT.

## *Завдання 10. Аналіз індустрій серед високооплачуваних віддалених працівників

In [22]:
print(df1['Industry'])

0                             Fintech
1        Retail and Consumer Services
2                Software Development
3        Retail and Consumer Services
4                Software Development
                     ...             
49186                             NaN
49187                             NaN
49188      Banking/Financial Services
49189                             NaN
49190                             NaN
Name: Industry, Length: 49191, dtype: object


In [23]:
#Вираховуємо 75-й перцентиль: Це число, вище якого знаходяться лише 25% "найбагатших" респондентів. 
quantile_75 = df1['ConvertedCompYearly'].quantile(0.75)
print(f'75-й перцентиль з компенсацій складає: {quantile_75}')

75-й перцентиль з компенсацій складає: 120596.0


In [26]:
salary_q75 = df1[
    (df1['ConvertedCompYearly'] >= quantile_75 ) & 
    (df1['RemoteWork'].str.lower() == 'remote')
    ]
print(salary_q75.head())

    ResponseId                      MainBranch              Age  \
29          30  I am a developer by profession  35-44 years old   
31          32  I am a developer by profession  35-44 years old   
41          42  I am a developer by profession  45-54 years old   
42          43  I am a developer by profession  35-44 years old   
48          49  I am a developer by profession  35-44 years old   

                                            EdLevel Employment  \
29  Master’s degree (M.A., M.S., M.Eng., MBA, etc.)   Employed   
31     Bachelor’s degree (B.A., B.S., B.Eng., etc.)   Employed   
41     Bachelor’s degree (B.A., B.S., B.Eng., etc.)   Employed   
42  Master’s degree (M.A., M.S., M.Eng., MBA, etc.)   Employed   
48     Bachelor’s degree (B.A., B.S., B.Eng., etc.)   Employed   

                                       EmploymentAddl  WorkExp  \
29        Engaged in paid work (10-19 hours per week)     19.0   
31                                  None of the above     17.0   
41

In [25]:
# Знаходжу індустрії для відфільтрованої групи
#value_counts() автоматично підрахує індустрії та відсортує їх
top_industries = salary_q75['Industry'].value_counts()
# Скидаю індекси, щоб отримати гарну таблицю
top_industries_df = top_industries.reset_index()
# Перейменовую стовпець
top_industries_df = top_industries_df.rename(columns={
    'count' : 'Count'
})

print('Топ індустрій для високооплачуваних віддалених працівників:')
print(top_industries_df)

Топ індустрій для високооплачуваних віддалених працівників:
                                      Industry  Count
0                         Software Development   1186
1                                      Fintech    190
2                                   Healthcare    188
3                                       Other:    176
4   Internet, Telecomm or Information Services    138
5                   Banking/Financial Services     88
6                                   Government     78
7                 Media & Advertising Services     75
8                 Retail and Consumer Services     65
9              Transportation, or Supply Chain     63
10        Computer Systems Design and Services     59
11                                   Insurance     46
12                                      Energy     45
13                               Manufacturing     33
14                            Higher Education     30


## Що ми бачимо з аналізу:
* Software Development — очікуваний лідер із величезним відривом (1186 осіб).
  Це база для високих зарплат на "ремоуті".
* Fintech та Healthcare йдуть майже нарівні.
  Це індустрії, де цінують безпеку та складні рішення, тому там готові платити вище за 75-й перцентиль.